# Monte Carlo Uncertainty Screening

This notebook explores how uncertainty in basic reservoir poroelastic properties propagates into an analytical subsidence estimate.

It uses the homogeneous **Geertsma solution without the Van Opstal rigid-basement correction** and is therefore intended as a screening model rather than as a replacement for the spatial Van Opstal calculation.

## 1. From Elastic Properties to Compaction

The drained constrained modulus is

$$
M=
\frac{E(1-\nu)}
{(1+\nu)(1-2\nu)}
$$

The uniaxial vertical compaction is

$$
\Delta h=
\frac{\alpha_B\,\Delta p\,h}{M}
$$

and the equivalent uniaxial compaction coefficient is

$$
C_m=\frac{\alpha_B}{M}
$$

so that

$$
\Delta h=C_m\Delta p\,h
$$

This provides the link between the sampled poroelastic parameters and the compaction coefficient used by the spatial model.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from reservoir_subsidence.uncertainty import (
    monte_carlo_center_subsidence,
)

## 2. Circular-Reservoir Screening Solution

For an idealized circular reservoir of radius $R$ at depth $z$,

$$
s=
2(1-\nu)\Delta h
\left(
1-\frac{z}{\sqrt{z^2+R^2}}
\right)
$$

This is the Geertsma centre-point solution for a uniformly compacting circular reservoir.

Subsidence is positive downward.

In [ ]:
reservoir_data = pd.read_csv(
    ROOT / "data" / "synthetic_uncertainty_reservoirs.csv"
)
reservoir_data

## 3. Monte Carlo Setup

The synthetic example uses independent uniform distributions:

- Young's modulus: 8–15 GPa
- Poisson's ratio: 0.20–0.30
- Biot coefficient: 0.70–1.00

The calculation uses **10,000 realizations** and a fixed random seed of **42**.

Pressure depletion, thickness, reservoir depth, and reservoir radius are deterministic in this screening example.

One sampled $E$–$\nu$–$\alpha_B$ triplet is applied to all reservoirs in each realization. This implies fully correlated reservoir mechanical properties in the current demonstration.

In [ ]:
simulations = monte_carlo_center_subsidence(
    reservoir_data,
    youngs_modulus_gpa_range=(8.0, 15.0),
    poisson_ratio_range=(0.20, 0.30),
    biot_coefficient_range=(0.70, 1.00),
    n_simulations=10_000,
    random_seed=42,
)

simulations.head()

## 4. Equivalent Compaction-Coefficient Distribution

In [ ]:
simulations[
    "equivalent_compaction_coefficient_per_bar"
].describe()

## 5. Subsidence Percentiles

In [ ]:
p10, p50, p90 = np.percentile(
    simulations["total_subsidence_mm"],
    [10, 50, 90],
)

summary = pd.Series(
    {
        "10th percentile [mm]": p10,
        "Median [mm]": p50,
        "90th percentile [mm]": p90,
        "Mean [mm]": simulations["total_subsidence_mm"].mean(),
        "Std. dev. [mm]": simulations["total_subsidence_mm"].std(),
    }
)

summary

## 6. Monte Carlo Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.hist(simulations["total_subsidence_mm"], bins=50)
ax.axvline(
    p10,
    linestyle="--",
    label=f"10th percentile = {p10:.1f} mm",
)
ax.axvline(
    p50,
    linestyle="--",
    label=f"Median = {p50:.1f} mm",
)
ax.axvline(
    p90,
    linestyle="--",
    label=f"90th percentile = {p90:.1f} mm",
)

ax.set_xlabel("Total Subsidence [mm]")
ax.set_ylabel("Number of Simulations")
ax.set_title("Monte Carlo Distribution of Predicted Subsidence")
ax.legend()
fig.tight_layout()

fig.savefig(
    ROOT / "figures" / "uncertainty_distribution.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

## 7. First-Pass Parameter Sensitivity

Spearman rank correlation is used as a simple indicator of the relative monotonic influence of the sampled parameters.

This does not replace a formal global sensitivity analysis.

In [ ]:
sensitivity = (
    simulations[
        [
            "youngs_modulus_gpa",
            "poisson_ratio",
            "biot_coefficient",
            "total_subsidence_mm",
        ]
    ]
    .corr(method="spearman")["total_subsidence_mm"]
    .drop("total_subsidence_mm")
    .sort_values(key=np.abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
sensitivity.plot(kind="barh", ax=ax)
ax.axvline(0.0, linewidth=1)
ax.set_xlabel("Spearman Rank Correlation with Subsidence")
ax.set_ylabel("")
ax.set_title("First-Pass Parameter Sensitivity")
fig.tight_layout()

fig.savefig(
    ROOT / "figures" / "uncertainty_sensitivity.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()

sensitivity

## 8. Relationship to the Spatial Model

The spatial model uses the Van Opstal rigid-basement correction and fixes $\nu=0.25$ because the implemented image-term coefficient set corresponds to that value.

The uncertainty-screening model uses the homogeneous Geertsma solution and allows $\nu$ to vary. The two workflows should therefore not be interpreted as directly equivalent predictions.

The purpose of this notebook is to:

- quantify how uncertainty in $E$, $\nu$, and $\alpha_B$ translates into uncertainty in $C_m$;
- identify which sampled parameters most strongly influence the analytical response;
- provide a basis for future uncertainty propagation through the full spatial model.

## 9. Limitations

- uniform distributions are assumptions rather than measurements;
- $E$, $\nu$, and $\alpha_B$ are sampled independently;
- reservoir geometry is idealized as circular;
- pressure depletion, thickness, depth, and radius are deterministic;
- rigid-basement effects are not represented in this notebook;
- dynamic elastic moduli from sonic measurements should not be used directly without an appropriate static conversion;
- the same sampled mechanical properties are applied to all reservoirs in each realization.

A future extension could sample reservoir-specific correlated property distributions and propagate them through the full spatial Van Opstal model.